# Week 6 — Solutions

In [ ]:
from pathlib import Path
import numpy as np
import torch
import torchvision
import torch.nn.functional as F
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
plt.style.use("../../assets/mplstyle/course.mplstyle")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

weights = torchvision.models.ResNet50_Weights.DEFAULT
model = torchvision.models.resnet50(weights=weights).to(DEVICE).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]
DATA = Path("../data")
IMG_PATHS = sorted(DATA.glob("*.jpg"))


def load_tensor(path):
    img = Image.open(path).convert("RGB").resize((224, 224))
    rgb = np.asarray(img).astype(np.float32) / 255.0
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    return img, rgb, x


## Solution 1 — Layer choice

In [ ]:
p = IMG_PATHS[0]
_, rgb, x = load_tensor(p)
with torch.no_grad():
    pred = int(model(x).argmax(-1))

layers = {"layer2": model.layer2[-1],
          "layer3": model.layer3[-1],
          "layer4": model.layer4[-1]}

fig, axes = plt.subplots(1, len(layers) + 1, figsize=(12, 3.2))
axes[0].imshow(rgb); axes[0].axis("off")
axes[0].set_title(f"{p.stem} → {categories[pred][:14]}", fontsize=9)
for ax, (name, layer) in zip(axes[1:], layers.items()):
    cam = GradCAM(model=model, target_layers=[layer])(
        input_tensor=x, targets=[ClassifierOutputTarget(pred)])[0]
    ax.imshow(show_cam_on_image(rgb, cam, use_rgb=True))
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()


**Reading.** Earlier layers (`layer2`) give **higher spatial resolution** but
**lower semantic specificity** — they highlight edges and textures that are not
class-specific. Later layers (`layer4`) give a coarser map but it actually corresponds
to the object the network classified. For a paper figure, `layer4[-1]` is the default;
`layer3` is a useful auxiliary view when the `layer4` map is uninformatively diffuse.


## Solution 2 — Insertion AUC

In [ ]:
def insertion_curve(image_path, n_steps=20):
    pil, rgb, x = load_tensor(image_path)
    with torch.no_grad():
        pred = int(model(x).argmax(-1))
    blurred = pil.filter(ImageFilter.GaussianBlur(radius=21))
    x_blur = preprocess(blurred).unsqueeze(0).to(DEVICE)

    cam = GradCAM(model=model, target_layers=[model.layer4[-1]])(
        input_tensor=x, targets=[ClassifierOutputTarget(pred)])[0]
    order = np.argsort(cam.flatten())[::-1]

    fracs = np.linspace(0, 1, n_steps + 1)
    ps = []
    for f in fracs:
        cur = x_blur.clone()
        n = int(f * len(order))
        if n > 0:
            idx_h, idx_w = np.unravel_index(order[:n], cam.shape)
            cur[0, :, idx_h, idx_w] = x[0, :, idx_h, idx_w]
        with torch.no_grad():
            ps.append(F.softmax(model(cur), dim=-1)[0, pred].item())
    return fracs, ps, pred


fig, ax = plt.subplots(figsize=(7, 4))
for p in IMG_PATHS:
    fr, ps, pred = insertion_curve(p)
    auc = np.trapz(ps, fr)
    ax.plot(fr, ps, label=f"{p.stem} ({categories[pred][:12]})  AUC={auc:.3f}")
ax.set(xlabel="fraction of top-CAM pixels inserted",
       ylabel="class probability",
       title="Insertion curves for Grad-CAM (higher = more faithful)")
ax.legend(loc="lower right", fontsize=8)
plt.show()


**Reading.** Insertion AUC asks "are the top-CAM pixels **sufficient** to recover
the prediction?" Deletion AUC asks "are they **necessary**?" Faithful explanations
score well on both; in practice the two metrics often disagree because models have
redundant features (deletion is shallow because something else takes over) or because
the prediction depends on context not captured in the CAM (insertion is shallow because
the highlighted patch alone isn't enough). Reporting both metrics is the standard.
